In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
# We encode the question to get its vector representation
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
v1

array([ 2.13904120e-02, -7.39799663e-02,  1.42068556e-03,  2.13816613e-02,
        2.45113149e-02,  3.15582752e-02, -1.10839702e-01, -1.05017498e-01,
       -6.18258938e-02, -6.42311852e-03,  3.72399064e-03,  9.06393602e-02,
       -9.49941017e-03,  6.53976873e-02,  1.10946558e-02, -2.10097339e-02,
       -3.35125513e-02, -4.31677327e-02,  9.96348727e-03,  1.41969863e-02,
       -6.40414879e-02, -7.04182126e-03, -7.91188031e-02,  5.80030754e-02,
        1.30212074e-03,  4.19733534e-03,  5.70979156e-02,  6.39447570e-02,
        2.49902904e-02, -3.95876691e-02, -3.79506797e-02,  2.70394552e-02,
        1.79423206e-02,  1.72272157e-02,  3.43311131e-02,  9.29055270e-03,
        5.86054735e-02, -4.97789569e-02, -5.05366083e-03,  4.34328541e-02,
       -1.56622957e-02, -2.97534503e-02, -5.13325352e-03,  5.13414629e-02,
        6.16060290e-03,  6.86980635e-02, -1.29505778e-02, -5.61938696e-02,
       -1.08265011e-02,  5.96683845e-02,  5.29939681e-02, -3.42755206e-02,
       -4.15274203e-02, -

In [3]:
# We encode our document, potentially relevant to the question, to get its vector representation
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

# Compare the two vectors using dot product
v1.dot(dv)
# We got 0.3233

np.float32(0.32332397)

In [4]:
# Now a definitely irrelevant query (to the document)
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)
# We got 0.0197

np.float32(0.019730438)

In [5]:
# The model 'all-MiniLM-L6-v2' outputs normalized vectors,
# when both vectors are normalized, the dot product equals cosine similarity.

In [6]:
# Now, we Embed our whole Dataset
# Since the ingest library that we created is in a different folder (../ingest), 
# we need to import it using relative imports

from pathlib import Path
import sys
import os

# 1. Get the absolute path of the directory containing this script
parent_dir = Path(os.getcwd()).resolve().parent

# 3. Add the parent directory to Python's search path
sys.path.append(str(parent_dir))

##############################################################################
# 4. Import the function normally
from ingest import load_faq_data

# Use the function
documents = load_faq_data()

In [7]:
documents[:5]

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

In [8]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

texts[:5]

["Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 "Course: What are the prerequisites for this course? To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is necessary. See [Readme on GitHub](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/README.md#prerequisites).\n\nIf you have these basics, you're ready to start — you don't need to master everything up

In [9]:
len(texts)

1350

In [10]:
# batch processing to avoid memory issues
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [11]:
import numpy as np
X = np.array(vectors)

# Next, how we retrieve relevant documents to augment our RAG

In [12]:
# When a query comes in, we embed it:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)
v_query

array([ 2.13904120e-02, -7.39799663e-02,  1.42068556e-03,  2.13816613e-02,
        2.45113149e-02,  3.15582752e-02, -1.10839702e-01, -1.05017498e-01,
       -6.18258938e-02, -6.42311852e-03,  3.72399064e-03,  9.06393602e-02,
       -9.49941017e-03,  6.53976873e-02,  1.10946558e-02, -2.10097339e-02,
       -3.35125513e-02, -4.31677327e-02,  9.96348727e-03,  1.41969863e-02,
       -6.40414879e-02, -7.04182126e-03, -7.91188031e-02,  5.80030754e-02,
        1.30212074e-03,  4.19733534e-03,  5.70979156e-02,  6.39447570e-02,
        2.49902904e-02, -3.95876691e-02, -3.79506797e-02,  2.70394552e-02,
        1.79423206e-02,  1.72272157e-02,  3.43311131e-02,  9.29055270e-03,
        5.86054735e-02, -4.97789569e-02, -5.05366083e-03,  4.34328541e-02,
       -1.56622957e-02, -2.97534503e-02, -5.13325352e-03,  5.13414629e-02,
        6.16060290e-03,  6.86980635e-02, -1.29505778e-02, -5.61938696e-02,
       -1.08265011e-02,  5.96683845e-02,  5.29939681e-02, -3.42755206e-02,
       -4.15274203e-02, -

In [13]:
# Next, we perform a matrix-vector multiplication
# The result is a vector of dot products, 
# one for each cosine similarity between the query and each document in our dataset
scores = X.dot(v_query)

# The highest score is the most similar document
idx = np.argmax(scores)
idx, scores[idx]
print('document index:', idx)
print('document:', documents[idx])

document index: 2
document: {'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}


In [14]:
# Usually, we want more than one best match,
# We can pull the top 5 match:
top5 = np.argsort(scores)[-5:]
# then we reverse the order to have the highest score first
top5 = top5[::-1]
print('document index:', top5)
print('scores:', scores[top5])

document index: [  2 625 907 538   7]
scores: [0.762941   0.7579371  0.7192132  0.6536312  0.56009996]


In [15]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192132
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

Now, we used the VectorSearch class in minsearch library to replicate everything above (with abstraction)

In [16]:
from minsearch import VectorSearch

# keyword_fields is can be used to subset the 'database' for faster search
# the minsearch.VectorSearch dose not do embedding, it only does the search, 
# so we need to provide the embeddings (X) and the documents
# It then bind the index together, and thus, provide some useful prebuilt methods.
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [17]:
# The API is the same as the one we used in chapter 1
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)
# Under the hood, this does the same things as we did above by hand.

results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [18]:
# With the keyword_fields, we can filter the search to only consider a subset of documents.
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

Now, we use the Vector Search for retrieval in our RAG system. Since we made the code very modular, this is as easy as plugging in the vector search to the RAGBase() class.

In [19]:
from dotenv import load_dotenv
from openai import OpenAI
import os
from rag_helper import RAGBase


load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(
    index=vindex,
    llm_client=openai_client,
)

In [20]:
query = 'I just found out about the program, can I still enroll?'
assistant.rag(query)

TypeError: VectorSearch.search() got an unexpected keyword argument 'boost_dict'

This is not working, as the search() method of the RAGBase() class was built with arguments and process specific to keyword search API.

We need to override the search() method to one that works with the VectorSearch API syntax.

In [24]:
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results = 5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}
        
        return self.index.search(query_vector, num_results=num_results, filter_dict = filter_dict)

In [25]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client
)

query = 'I just found out about the program, can I still enroll?'
vector_assistant.rag(query)

'Unfortunately, based on the information provided, you are too late to enroll for this specific program\'s next cohort, and you will have to wait until Summer 2027 for the next offering. \n\nHowever, you can still join and participate in the current cohort. Since registration is not a requirement, you can start learning and submitting homework while the form is still open. \n\nIf you\'re interested in receiving a certificate, you should submit your project while the program is still accepting submissions and participate in the "live" cohort.'